[![Abrir en Google Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/gsilvaoelker/campos_ondas_electromagneticas/blob/main/unidad_01/04_continuidad_maxwell_condiciones_de_borde.ipynb)

Pulse el botón para ejecutar este notebook en Google Colab sin instalar nada.

# Semana 4 — Continuidad, Maxwell y condiciones de borde

**Campos y Ondas Electromagnéticas (ICEE1033) — Unidad 1**

## 1. Objetivos de aprendizaje

Al terminar este notebook usted podrá:

1. Deducir el tiempo de relajación $\tau = \varepsilon/\sigma$ combinando
   continuidad, Gauss y Ohm.
2. Predecir cuánto tarda en desaparecer una carga puesta dentro de un
   material conductor.
3. Aplicar las condiciones de borde entre dos dieléctricos.
4. Decir qué componente del campo se conserva y cuál salta al cruzar una
   interfaz.

In [ ]:
# Preparación del entorno: funciona igual en Google Colab y en una copia local.
import sys
import urllib.request
from pathlib import Path

MODULOS = ["utilidades_notebook.py", "constantes_fisicas.py", "medios_y_condiciones_de_borde.py"]
URL_SRC = (
    "https://raw.githubusercontent.com/"
    "gsilvaoelker/campos_ondas_electromagneticas/main/src/"
)

# Se exige que estén *todos* los módulos, no solo la carpeta: así, si otro
# notebook ya creó `src/` en esta misma sesión, igual se descarga lo que falte.
raiz = next(
    (p for p in [Path.cwd(), *Path.cwd().parents]
     if all((p / "src" / modulo).exists() for modulo in MODULOS)),
    None,
)
if raiz is None:  # Google Colab: descargar los módulos del curso.
    raiz = Path.cwd()
    (raiz / "src").mkdir(exist_ok=True)
    for modulo in MODULOS:
        if not (raiz / "src" / modulo).exists():
            urllib.request.urlretrieve(URL_SRC + modulo, raiz / "src" / modulo)
sys.path.insert(0, str(raiz / "src"))

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from medios_y_condiciones_de_borde import (
    tiempo_de_relajacion,
    densidad_de_carga_relajacion,
    divergencia_corriente_relajacion,
    campo_transmitido,
    densidad_flujo_normal,
)
from utilidades_notebook import configurar_estilo_graficos, tabla_resultados

configurar_estilo_graficos()

## 2. De dónde sale todo esto

### 2.1 Tres ecuaciones conocidas dan un resultado nuevo

Junte la ecuación de continuidad, la ley de Ohm y la ley de Gauss:

$$
\nabla\cdot\mathbf{J} = -\frac{\partial \rho_v}{\partial t}
\;\Longrightarrow\;
\sigma\,\nabla\cdot\mathbf{E} = -\frac{\partial \rho_v}{\partial t}
\;\Longrightarrow\;
\frac{\sigma}{\varepsilon}\rho_v = -\frac{\partial \rho_v}{\partial t}.
$$

El resultado es una ecuación diferencial cuya solución es una exponencial
decreciente.

Lo notable es que **no supusimos nada sobre la forma de la carga**. Cualquier
carga libre dentro de un material óhmico homogéneo desaparece con la misma
constante de tiempo, sea una bola, un cubo o una mancha irregular.

### 2.2 Las condiciones de borde no son ecuaciones nuevas

Son las ecuaciones de Maxwell aplicadas a una superficie donde el material
cambia de golpe.

La condición sobre $\mathbf{E}$ tangencial sale de aplicar la ley de Faraday
a un lazo pequeñito que cruza la interfaz. La condición sobre $\mathbf{D}$
normal sale de aplicar la ley de Gauss a una caja pequeñita que la cruza.

Nada nuevo: solo Maxwell mirado de cerca.

## 3. Ecuaciones

**Ecuación de continuidad:**

$$
\nabla\cdot\mathbf{J} = -\frac{\partial\rho_v}{\partial t}.
$$

**Las cuatro de Maxwell:**

$$
\begin{aligned}
\nabla\cdot\mathbf{D} &= \rho_v, &
\nabla\times\mathbf{E} &= -\frac{\partial\mathbf{B}}{\partial t}, \\
\nabla\cdot\mathbf{B} &= 0, &
\nabla\times\mathbf{H} &= \mathbf{J} + \frac{\partial\mathbf{D}}{\partial t}.
\end{aligned}
$$

**Relajación de carga:**

$$
\tau = \frac{\varepsilon}{\sigma} = \frac{\varepsilon_0\varepsilon_r}{\sigma},
\qquad
\rho_v(t) = \rho_0\,e^{-t/\tau},
\qquad
\nabla\cdot\mathbf{J} = \frac{\rho_v(t)}{\tau}.
$$

**Condiciones de borde** entre dos dieléctricos sin carga libre en la
superficie:

$$
E_{1t} = E_{2t}
\qquad\text{(lo tangencial de } \mathbf{E}\text{ se conserva)},
$$

$$
D_{1n} = D_{2n}
\quad\Longrightarrow\quad
E_{2n} = \frac{\varepsilon_{r1}}{\varepsilon_{r2}} E_{1n}
\qquad\text{(lo normal de } \mathbf{E}\text{ salta)}.
$$

## 4. Qué significa físicamente

**La carga se escapa a la superficie.** Dentro de un conductor la carga libre
se repele a sí misma y migra hacia el borde. $\tau$ mide cuánto tarda. En el
cobre son unos $10^{-19}$ s: instantáneo para cualquier propósito práctico.
En un buen aislante pueden ser minutos, y por eso un plástico frotado se queda
cargado.

**El material de este notebook es un caso intermedio.** Con
$\varepsilon_r = 4.0$ y $\sigma = 2\times10^{-8}$ S/m —un aislante mediocre,
tipo plástico húmedo— el tiempo es de milisegundos: lento para medirlo,
rápido para no ser electrostática.

**Al cruzar, el campo se dobla.** Lo tangencial pasa intacto y lo normal se
achica. El resultado es que el vector $\mathbf{E}$ cambia de dirección,
exactamente igual que un rayo de luz al refractarse. De hecho es el mismo
fenómeno: en la semana 10 volveremos a encontrarlo con el nombre de ley de
Snell.

## 5. Parámetros modificables

Esta es la única celda que conviene editar. Cambie un valor, ejecute el notebook
completo y compare con lo que tenía antes.

In [ ]:
# --- Problema 1: relajación de carga ---
eps_r_medio = 4.0            # permitividad relativa del medio
conductividad = 2.0e-8       # conductividad sigma [S/m]
densidad_inicial = 5.0e-6    # rho_0 puesta en t = 0 [C/m^3]
tiempo_evaluacion = 3.0e-3   # instante donde evaluar rho_v [s]

# --- Problema 2: interfaz entre dos dieléctricos ---
eps_r1 = 2.0   # permitividad relativa del medio 1
eps_r2 = 5.0   # permitividad relativa del medio 2

# Campo en el medio 1 [V/m]. La interfaz es el plano z = 0, así que
# la componente z es la normal y las componentes x e y son tangenciales.
E1 = np.array([3.0, 0.0, 4.0]) * 1.0e3

## 6. Implementación

### 6.1 Problema 1 — la carga que se escapa

In [ ]:
tau = tiempo_de_relajacion(eps_r_medio, conductividad)
rho_en_t = float(densidad_de_carga_relajacion(tiempo_evaluacion, densidad_inicial, tau))
divergencia_J = divergencia_corriente_relajacion(rho_en_t, tau)

### 6.2 Problema 2 — el campo que se dobla

In [ ]:
EJE_NORMAL = 2  # la interfaz es el plano z = 0

E2 = campo_transmitido(E1, eps_r1, eps_r2, eje_normal=EJE_NORMAL)

D1_normal = densidad_flujo_normal(E1[EJE_NORMAL], eps_r1)
D2_normal = densidad_flujo_normal(E2[EJE_NORMAL], eps_r2)

## 7. Resultados numéricos

In [ ]:
tabla_resultados(
    [
        ("Tiempo de relajación", "tau", tau, "s"),
        ("Densidad de carga en t", "rho_v(t)", rho_en_t, "C/m^3"),
        ("Fracción que queda", "rho_v(t)/rho_0", rho_en_t / densidad_inicial, "-"),
        ("Divergencia de la corriente", "div(J)", divergencia_J, "A/m^3"),
    ]
)

In [ ]:
tabla_resultados(
    [
        ("Campo en el medio 1, tangencial", "E_1t", E1[0], "V/m"),
        ("Campo en el medio 2, tangencial", "E_2t", E2[0], "V/m"),
        ("Campo en el medio 1, normal", "E_1n", E1[EJE_NORMAL], "V/m"),
        ("Campo en el medio 2, normal", "E_2n", E2[EJE_NORMAL], "V/m"),
        ("Flujo en el medio 1, normal", "D_1n", D1_normal, "C/m^2"),
        ("Flujo en el medio 2, normal", "D_2n", D2_normal, "C/m^2"),
        ("Salto de D_n", "D_2n - D_1n", D2_normal - D1_normal, "C/m^2"),
    ]
)

Ángulos que forma el campo con la normal a cada lado:

In [ ]:
angulo_1 = np.rad2deg(np.arctan2(np.hypot(E1[0], E1[1]), E1[EJE_NORMAL]))
angulo_2 = np.rad2deg(np.arctan2(np.hypot(E2[0], E2[1]), E2[EJE_NORMAL]))
print(f"Ángulo con la normal en el medio 1 = {angulo_1:.3f} grados")
print(f"Ángulo con la normal en el medio 2 = {angulo_2:.3f} grados")
print(f"tan(theta_2) / tan(theta_1) = "
      f"{np.tan(np.deg2rad(angulo_2)) / np.tan(np.deg2rad(angulo_1)):.6f}")
print(f"eps_r2 / eps_r1             = {eps_r2 / eps_r1:.6f}")

## 8. Visualización

Cómo se apaga la carga. La línea vertical marca $t = \tau$, donde queda el
$1/e \approx 36.8\,\%$.

In [ ]:
t = np.linspace(0.0, 6.0 * tau, 300)
rho = densidad_de_carga_relajacion(t, densidad_inicial, tau)

fig, eje = plt.subplots()
eje.plot(t * 1.0e3, rho * 1.0e6)
eje.axvline(tau * 1.0e3, color="black", linestyle="--", label="t = tau")
eje.scatter([tiempo_evaluacion * 1.0e3], [rho_en_t * 1.0e6], color="black",
            zorder=5, label="instante evaluado")
eje.set_xlabel("t (ms)")
eje.set_ylabel("rho_v (microC/m^3)")
eje.set_title("La carga libre se escapa del interior")
eje.legend()
fig.tight_layout()
plt.show()

## 9. Qué nos dicen los resultados

**La geometría no aparece por ninguna parte.** $\tau = \varepsilon/\sigma$
solo contiene propiedades del material. Una esfera, un cubo o una lámina del
mismo material pierden su carga interior a la misma velocidad.

**A las cinco constantes de tiempo ya no queda nada.** En $t = 5\tau$ sobra
menos del 0.7 %. Ésa es la regla práctica que justifica tratar el interior de
un conductor como neutro.

**Lo tangencial pasa intacto y lo normal se achica.** En la tabla,
$E_{1t} = E_{2t}$ exactamente, mientras que $E_{2n}$ es 0.4 veces $E_{1n}$.
Al mismo tiempo $D_{2n} - D_{1n} = 0$ hasta la precisión de la máquina: es la
comprobación de que no hay carga libre en la interfaz.

**El campo se refracta.** Los ángulos cumplen
$\tan\theta_2/\tan\theta_1 = \varepsilon_{r2}/\varepsilon_{r1}$. Al pasar a un
material de mayor permitividad, el campo se aleja de la normal.

## 10. Ejercicios para experimentar

            1. Cambie `conductividad` a `5.8e7` (cobre). ¿Cuánto vale $\tau$? ¿Se alcanza
               a ver el decaimiento en la escala de milisegundos del gráfico?
            2. Cambie `conductividad` a `1.0e-14` (teflón). ¿Cuánto tarda la carga en caer
               al 1 %? Exprese el resultado en unidades que se entiendan.
            3. Ponga `tiempo_evaluacion` en `tau`, después en `3 * tau` y en `5 * tau`.
               Compruebe la regla de las cinco constantes de tiempo.
            4. Intercambie `eps_r1` y `eps_r2`. ¿El campo se acerca o se aleja de la
               normal?
            5. Ponga `E1 = np.array([0.0, 0.0, 4.0]) * 1.0e3` (incidencia normal). ¿Cuánto
               valen los ángulos? ¿Hay refracción?
            6. Ponga `E1 = np.array([3.0, 0.0, 0.0]) * 1.0e3` (campo puramente
               tangencial). ¿Cambia algo al cruzar? ¿Por qué?